# Corpus-regression MaxRL

In [ ]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.corpus_regression.maxrl import CorpusRegressionMaxRLConfig
from src.experiments.corpus_regression.utils import dim_averaged_metrics_from_parquet

repo_root = get_repo_base()
device = torch.device("cuda:0")

### Configs

In [ ]:
config = CorpusRegressionMaxRLConfig.get_canonical(
    dataset_base_folder=repo_root / "artifacts" / "corpus-regression",
    study_base_folder=repo_root / "artifacts" / "corpus-regression-maxrl-example",
    num_lookforward_tokens=1,
    train_epochs=2,
    num_rollouts_per_sample=4,
    gaussian_stdev=1.0,
    subtract_baseline=True,
)

display(config.visualize())

In [ ]:
state = config.initialize(device=device)
state.run_training()

### Results

In [ ]:
metrics = pl.read_parquet(config.study_folder / "metrics.parquet")
metrics

In [ ]:
dim_averaged_metrics_from_parquet(metrics, split="train").join(
    dim_averaged_metrics_from_parquet(metrics, split="val"), on="epoch"
)

In [ ]:
last_epoch = int(metrics["epoch"].max())
validation_df = pl.read_parquet(
    config.study_folder / str(last_epoch) / "validation.parquet"
)
validation_df.head()